# E/S de Alto Desempeño con HDF5: Checkpointing de un Escaneo MPI

*Métodos Computacionales Modernos para la Física — Módulo II: Fundamentos de Alto Desempeño*

El cuaderno anterior (`mpi_balanceo_carga.ipynb`) terminó con un escaneo de
parámetros repartido con el patrón gerente-trabajador, eficiente incluso con
costo por punto muy desigual -- pero guardando todo en memoria. Hoy ese
mismo escaneo aprende a sobrevivir una caída: cada resultado se escribe a un
archivo HDF5 en cuanto llega, y una corrida interrumpida se puede reanudar
exactamente donde se quedó, sin recalcular nada.

Como en los dos cuadernos anteriores, el escaneo real corre como script
externo vía `mpirun` -- un programa MPI no vive dentro de un solo kernel de
Jupyter. Esta vez además se escribe la salida a un archivo en disco en vez
de capturarla por una tubería (`subprocess.PIPE`) y se busca el marcador de
resultado en cualquier parte de cada línea, no sólo al inicio -- dos
lecciones de robustez que el cuaderno de la Sesión 10 tuvo que aprender por
las malas (ver esa sesión para el detalle).

In [1]:
import sys, os, math, time, subprocess, textwrap, tempfile
import numpy as np
import h5py
import mpi4py
from mpi4py import MPI

print(f"Python  {sys.version.split()[0]}")
print(f"NumPy   {np.__version__}")
print(f"h5py    {h5py.__version__}  (HDF5 {h5py.version.hdf5_version})")
print(f"mpi4py  {mpi4py.__version__}  ({MPI.get_vendor()[0]} {'.'.join(map(str, MPI.get_vendor()[1]))})")
print(f"HDF5 compilado con soporte MPI (paralelo): {h5py.get_config().mpi}")

MCA = ['--mca', 'btl', 'self,vader']
GRID_N = 24

def run_mpi(args, nproc, timeout=120):
    '''Corre un script externo con mpirun, redirigiendo a archivo (no PIPE) --
    ver sesion10.md, Seccion 4, sobre por que PIPE fallo aqui.'''
    with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.log') as f:
        outpath = f.name
    with open(outpath, 'w') as fout:
        proc = subprocess.run(['mpirun', *MCA, '-np', str(nproc), sys.executable, *args],
                               stdout=fout, stderr=subprocess.STDOUT, timeout=timeout)
    with open(outpath) as f:
        out = f.read()
    os.unlink(outpath)
    return proc.returncode, out

def find_result(out):
    '''Busca RESULT|... en cualquier parte de la linea -- mpirun a veces
    antepone un codigo ANSI de otro rango justo antes del marcador.'''
    for line in out.splitlines():
        if 'RESULT|' in line:
            return line[line.index('RESULT|'):].split('|')
    return None

Python  3.14.0
NumPy   2.3.4
h5py    3.16.0  (HDF5 1.14.6)
mpi4py  4.1.2  (Open MPI 5.0.8)
HDF5 compilado con soporte MPI (paralelo): False


## 1. `h5py` en lo esencial

Antes del escaneo real, un archivo de juguete: datasets de tamaño fijo,
`fillvalue` como marcador de "todavía no calculado", y un atributo.

In [2]:
demo_path = 'demo.h5'
if os.path.exists(demo_path):
    os.remove(demo_path)

with h5py.File(demo_path, 'a') as f:
    f.create_dataset('x', data=np.linspace(0, 1, 5))
    f.create_dataset('y', shape=(5,), dtype='f8', fillvalue=np.nan)
    f.attrs['creado_por'] = 'sesion11 demo'
    f['y'][2] = 3.14  # solo se llena una posicion

with h5py.File(demo_path, 'r') as f:
    print('x:', f['x'][:])
    print('y:', f['y'][:], '  (NaN = todavia no calculado)')
    print('atributo:', f.attrs['creado_por'])

os.remove(demo_path)

x: [0.   0.25 0.5  0.75 1.  ]
y: [ nan  nan 3.14  nan  nan]   (NaN = todavia no calculado)
atributo: sesion11 demo


## 2. El escaneo con checkpointing

Se reutiliza, sin cambios de física, el modelo `higgs_portal_sigma`
compilado en la Sesión 10 -- el que mostró costo por punto muy desigual
(hasta 27x) cruzando la resonancia. La única adición real es que el gerente,
además de guardar cada resultado en memoria, lo escribe al archivo HDF5 y
hace `flush()` antes de pedir el siguiente.

In [3]:
script = textwrap.dedent(r'''
    import sys, os, math, time
    import numpy as np
    import h5py
    from numba import njit
    from scipy.integrate import solve_ivp
    from scipy.special import kn
    from mpi4py import MPI

    M_HIGGS, V_EWSB, WIDTH_HIGGS = 125.1, 246.22, 4.07e-3

    @njit(cache=True)
    def higgs_portal_sigma_njit(s, lam_hs, m_dm):
        denom = (s - M_HIGGS**2)**2 + (M_HIGGS*WIDTH_HIGGS)**2
        sqrt_s = math.sqrt(s)
        gam = 0.0
        m_b = 4.18
        if sqrt_s > 2*m_b:
            gam += (3*sqrt_s/(16*math.pi))*(m_b**2/V_EWSB**2)*(1-4*m_b**2/sqrt_s**2)**1.5
        m_w = 80.38
        if sqrt_s > 2*m_w:
            x = 4*m_w**2/sqrt_s**2
            gam += (sqrt_s**3/(32*math.pi*V_EWSB**2))*math.sqrt(1-x)*(1-x+0.75*x**2)
        m_z = 91.19
        if sqrt_s > 2*m_z:
            x = 4*m_z**2/sqrt_s**2
            gam += (sqrt_s**3/(64*math.pi*V_EWSB**2))*math.sqrt(1-x)*(1-x+0.75*x**2)
        m_t = 173.0
        if sqrt_s > 2*m_t:
            gam += (3*sqrt_s/(16*math.pi))*(m_t**2/V_EWSB**2)*(1-4*m_t**2/sqrt_s**2)**1.5
        flux = math.sqrt(s*(s - 4*m_dm**2))
        return (2*lam_hs**2*V_EWSB**2/denom)*(gam/flux)

    @njit(cache=True)
    def _bessk1e(x):
        if x <= 2.0:
            t = (x/3.75)**2
            i1 = x*(0.5 + t*(0.87890594 + t*(0.51498869 + t*(0.15084934 + t*(0.02658733 + t*(0.00301532 + t*0.00032411))))))
            y = x*x/4.0
            k1 = (math.log(x/2.0)*i1) + (1.0/x)*(1.0 + y*(0.15443144 + y*(-0.67278579 +
                 y*(-0.18156897 + y*(-0.01919402 + y*(-0.00110404 + y*(-0.00004686)))))))
            return k1 * math.exp(x)
        else:
            y = 2.0/x
            return (1.0/math.sqrt(x)) * (1.25331414 + y*(0.23498619 + y*(-0.0365562 +
                   y*(0.01504268 + y*(-0.00780353 + y*(0.00325614 + y*(-0.00068245)))))))

    @njit(cache=True)
    def _bessk0e(x):
        if x <= 2.0:
            t = (x/3.75)**2
            i0 = 1.0 + t*(3.5156229 + t*(3.0899424 + t*(1.2067492 + t*(0.2659732 + t*(0.0360768 + t*0.0045813)))))
            y = x*x/4.0
            k0 = (-math.log(x/2.0)*i0) + (-0.57721566 + y*(0.42278420 + y*(0.23069756 +
                 y*(0.0348859 + y*(0.00262698 + y*(0.0001075 + y*0.0000074))))))
            return k0 * math.exp(x)
        else:
            y = 2.0/x
            return (1.0/math.sqrt(x)) * (1.25331414 + y*(-0.07832358 + y*(0.02189568 +
                   y*(-0.01062446 + y*(0.00587872 + y*(-0.0025154 + y*0.00053208))))))

    @njit(cache=True)
    def _bessk2e(x):
        return _bessk0e(x) + (2.0/x) * _bessk1e(x)

    _N_NODES = 64
    _GL_X, _GL_W = np.polynomial.legendre.leggauss(_N_NODES)

    @njit(cache=True)
    def _thermal_average_gl_higgs(m, T, lam_hs, m_dm, nodes, weights):
        s_min = 4.0 * m * m
        s_max = (2.0*m + 30.0*T)**2
        half = 0.5 * (s_max - s_min)
        mid  = 0.5 * (s_max + s_min)
        total = 0.0
        for i in range(nodes.shape[0]):
            s = half * nodes[i] + mid
            if s > s_min:
                sqrt_s = math.sqrt(s)
                arg = (sqrt_s - 2.0*m) / T
                exp_factor = math.exp(-arg) if arg < 100.0 else 0.0
                sigma_val = higgs_portal_sigma_njit(s, lam_hs, m_dm)
                val = sigma_val * (s - s_min) * sqrt_s * _bessk1e(sqrt_s / T) * exp_factor
            else:
                val = 0.0
            total += weights[i] * val
        integral = half * total
        k2 = _bessk2e(m / T)
        return 0.0 if k2 <= 0.0 else integral / (8.0 * m**4 * T * k2 * k2)

    M_PLANCK = 1.2209e19
    RHO_CRIT_OVER_H2 = 1.0537e-5 * ((5.0677e13)**-3)
    S_NOW = 2891.2 * ((5.0677e13)**-3)
    G_STAR = 106.75

    def y_eq(x, m):
        return (45.0/(4*np.pi**4)) * (2.0/G_STAR) * (x**2) * kn(2, x)

    def calculate_omega_h2_higgs(m_dm, lam_hs, x_start=1.0, x_end=1000.0):
        def rhs(x, logy):
            T = m_dm/x
            h = 1.66*np.sqrt(G_STAR)*(T**2)/M_PLANCK
            s_ent = (2*np.pi**2/45)*G_STAR*(T**3)
            sigv = _thermal_average_gl_higgs(m_dm, T, lam_hs, m_dm, _GL_X, _GL_W)
            y_eq_val = y_eq(x, m_dm)
            if y_eq_val <= 0:
                return [-(s_ent*sigv/(h*x))*math.exp(logy[0])]
            diff = logy[0] - math.log(y_eq_val)
            if diff > 50:
                return [-(s_ent*sigv/(h*x))*math.exp(logy[0])]
            return [-(s_ent*sigv/(h*x))*2*y_eq_val*math.sinh(diff)]
        y0 = y_eq(x_start, m_dm)
        sol = solve_ivp(rhs, [x_start, x_end], [math.log(y0)], method='Radau', rtol=1e-6, atol=1e-12)
        return m_dm * S_NOW * math.exp(sol.y[0][-1]) / RHO_CRIT_OVER_H2

    def scan_point(m_s):
        return calculate_omega_h2_higgs(m_s, 0.01)

    comm = MPI.COMM_WORLD
    rank, size = comm.Get_rank(), comm.Get_size()
    GRID = np.linspace(40.0, 85.0, 24)
    CKPT_FILE = sys.argv[1]
    CRASH_AFTER = int(sys.argv[2]) if len(sys.argv) > 2 else -1

    _thermal_average_gl_higgs(60.0, 5.0, 0.01, 60.0, _GL_X, _GL_W)  # calentamiento

    TAG_WORK, TAG_RESULT, TAG_DONE = 1, 2, 3

    if rank == 0:
        t0 = MPI.Wtime()
        resumed = os.path.exists(CKPT_FILE)
        f = h5py.File(CKPT_FILE, 'a')
        if 'omega_h2' not in f:
            f.create_dataset('m_grid', data=GRID)
            f.create_dataset('omega_h2', shape=(len(GRID),), dtype='f8', fillvalue=np.nan)
            f.create_dataset('done', shape=(len(GRID),), dtype='bool', fillvalue=False)
            f.attrs['grid_description'] = 'm_S en GeV, lambda_hs=0.01, higgs_portal_sigma'
        done_mask = f['done'][:]
        n_previos = int(done_mask.sum())
        print(f"checkpoint {'reanudado' if resumed else 'nuevo'}: {n_previos}/{len(GRID)} puntos ya en disco", flush=True)

        task_queue = [i for i in range(len(GRID)) if not done_mask[i]]
        n_nuevos = 0
        for w in range(1, size):
            if task_queue:
                idx = task_queue.pop(0)
                comm.send((idx, GRID[idx]), dest=w, tag=TAG_WORK)
            else:
                comm.send(None, dest=w, tag=TAG_DONE)

        pending = size - 1
        while pending > 0:
            status = MPI.Status()
            msg = comm.recv(source=MPI.ANY_SOURCE, tag=MPI.ANY_TAG, status=status)
            if status.Get_tag() == TAG_RESULT:
                idx, val = msg
                f['omega_h2'][idx] = val
                f['done'][idx] = True
                f.flush()
                n_nuevos += 1
                if CRASH_AFTER > 0 and n_nuevos >= CRASH_AFTER:
                    print(f"SIMULANDO CAIDA tras {n_nuevos} puntos nuevos", flush=True)
                    f.close()
                    comm.Abort(1)
                worker = status.Get_source()
                if task_queue:
                    idx2 = task_queue.pop(0)
                    comm.send((idx2, GRID[idx2]), dest=worker, tag=TAG_WORK)
                else:
                    comm.send(None, dest=worker, tag=TAG_DONE)
                    pending -= 1
            elif status.Get_tag() == TAG_DONE:
                pending -= 1

        t1 = MPI.Wtime()
        print(f"RESULT|{size}|{t1-t0:.6f}|{n_nuevos}")
        f.close()
    else:
        while True:
            status = MPI.Status()
            msg = comm.recv(source=0, tag=MPI.ANY_TAG, status=status)
            if status.Get_tag() == TAG_DONE:
                break
            idx, m = msg
            comm.send((idx, scan_point(m)), dest=0, tag=TAG_RESULT)
''')
with open('_mpi_checkpoint.py', 'w') as f:
    f.write(script)
print("script escrito")

script escrito


## 3. Simulando una caída a la mitad del escaneo

`CRASH_AFTER=10`: el gerente llama `comm.Abort()` -- la terminación más
abrupta que MPI ofrece, deliberadamente, para no darle a la demostración
ninguna ventaja de cierre ordenado -- después de escribir el décimo punto
nuevo.

In [4]:
CKPT = 'scan_checkpoint.h5'
if os.path.exists(CKPT):
    os.remove(CKPT)

rc, out = run_mpi(['_mpi_checkpoint.py', CKPT, '10'], nproc=5)
print(out)
print(f"codigo de salida (se espera != 0, la caida es deliberada): {rc}")

checkpoint nuevo: 0/24 puntos ya en disco
SIMULANDO CAIDA tras 10 puntos nuevos
--------------------------------------------------------------------------
MPI_ABORT was invoked on rank 0 in communicator MPI_COMM_WORLD
  Proc: [[31509,1],0]
  Errorcode: 1

NOTE: invoking MPI_ABORT causes Open MPI to kill all MPI processes.
You may or may not see output from other processes, depending on
exactly when Open MPI kills them.
--------------------------------------------------------------------------
--------------------------------------------------------------------------
prterun has exited due to process rank 0 with PID 0 on node Lenny calling
"abort". This may have caused other processes in the application to be
terminated by signals sent by prterun (as reported here).
--------------------------------------------------------------------------
 
codigo de salida (se espera != 0, la caida es deliberada): 1


In [5]:
with h5py.File(CKPT, 'r') as f:
    n_done = int(f['done'][:].sum())
    print(f"Tras la caida -- puntos en disco: {n_done} / {len(f['done'])}")
    assert n_done == 10, "se esperaban exactamente 10 puntos antes de la caida simulada" 

Tras la caida -- puntos en disco: 10 / 24


## 4. Reanudando

Mismo comando, mismo archivo -- sin indicarle al script nada especial sobre
qué reanudar. Eso lo decide el propio archivo.

In [6]:
rc, out = run_mpi(['_mpi_checkpoint.py', CKPT], nproc=5)
print(out)

_, _, t_resume, n_nuevos_resume = find_result(out)
print(f"Puntos nuevos calculados al reanudar: {n_nuevos_resume}  (se esperan {GRID_N - 10})")
assert int(n_nuevos_resume) == GRID_N - 10

checkpoint reanudado: 10/24 puntos ya en disco
RESULT|5|1.448067|14

Puntos nuevos calculados al reanudar: 14  (se esperan 14)


In [7]:
with h5py.File(CKPT, 'r') as f:
    done = f['done'][:]
    omega = f['omega_h2'][:]
    print(f"Completos: {done.sum()} / {len(done)}")
    print(f"NaN restantes: {np.isnan(omega).sum()}")
    assert done.all() and not np.isnan(omega).any(), "el escaneo deberia estar completo y sin NaN"
print("Escaneo verificado: completo, consistente, sin recalcular los primeros 10 puntos.")

Completos: 24 / 24
NaN restantes: 0
Escaneo verificado: completo, consistente, sin recalcular los primeros 10 puntos.


## 5. ¿Cuánto cuesta checkpointear?

Una corrida limpia, sin interrupción, con el mismo checkpointing activo en
cada punto -- comparada directamente contra el resultado sin checkpointing
de la Sesión 10 (`5.525 s`, mismos 24 puntos, mismos 4 trabajadores).

In [8]:
CKPT_CLEAN = 'scan_clean.h5'
if os.path.exists(CKPT_CLEAN):
    os.remove(CKPT_CLEAN)

rc, out = run_mpi(['_mpi_checkpoint.py', CKPT_CLEAN], nproc=5)
_, size, t_clean, n_clean = find_result(out)
t_clean = float(t_clean)

t_sesion10_sin_checkpoint = 5.525  # Sesion 10, Seccion 4: dinamico, 1 gerente + 4 trabajadores

print(f"Con checkpointing (hoy):        {t_clean:.3f} s")
print(f"Sin checkpointing (Sesion 10):  {t_sesion10_sin_checkpoint:.3f} s")
print(f"Diferencia: {100*(t_clean - t_sesion10_sin_checkpoint)/t_sesion10_sin_checkpoint:+.1f}%  (dentro del ruido normal de la maquina)")

os.remove(CKPT_CLEAN)
os.remove(CKPT)
os.remove('_mpi_checkpoint.py')

Con checkpointing (hoy):        5.486 s
Sin checkpointing (Sesion 10):  5.525 s
Diferencia: -0.7%  (dentro del ruido normal de la maquina)


## 6. Para llevar

* HDF5 resuelve un problema concreto que un archivo de texto o `pickle` no
  resuelven bien a la escala del escaneo 5D de referencia: anexar sin
  reescribir todo, leer una posición sin cargar el archivo completo, y
  llevar sus propios metadatos (Sección 1).
* El patrón gerente-trabajador de la Sesión 10 ya era, sin haberlo planeado
  así, la arquitectura correcta para esto: un único escritor, que es
  exactamente lo que esta instalación de HDF5 (sin soporte MPI/paralelo,
  `h5py.get_config().mpi == False`) exige (Sección 2).
* `fillvalue=np.nan` convierte al propio archivo en el registro de qué
  falta -- reanudar es, literalmente, filtrar por `done == False`
  (Secciones 2-4), sin necesitar ningún mecanismo externo de seguimiento de
  progreso.
* Se demostró, no se prometió: una caída forzada con `MPI_ABORT` a la mitad
  del escaneo dejó un archivo completamente recuperable, y la segunda
  corrida calculó exactamente los puntos que faltaban -- ni uno más
  (Secciones 3-4, con aserciones que lo verifican, no solo lo imprimen).
* El costo de checkpointear, medido, es indistinguible del ruido de la
  máquina en este problema (Sección 5) -- porque cada punto cuesta cientos
  de milisegundos a segundos de cómputo, y un `flush()` cuesta
  microsegundos. No hay excusa de desempeño para omitirlo.
* Lo que sigue sin resolverse -- todo lo del módulo hasta ahora corrió sobre
  CPU -- es el tema de cierre del Módulo II: GPUs, en la Sesión 12.